# 03 — Method C: **INT4 QAT** (양자화 인식 학습, 오차 보정)

> ## ⚙️ 실행 모드 배너 — 이 노트북은 **Azure A100 80GB(japaneast, Spot)에서 실제 실행**됨
> base=**Qwen3-8B**, transformers HF 백엔드, BF16 LoRA + TorchAO int4 tile-packed.
> 학습/평가는 `quantization/v2_run`이 seed 42·43·44로 이미 수행 — 이 노트북은 **실 아티팩트 로드 +
> 1회 생성 데모 + 집계 수치 표시**(수 시간 재학습 없음).

### 이 방법(C) — 무엇/왜/어떻게
- **무엇**: A 머지에서 출발해 **양자화를 인식하며(full-param STE)** 재학습 후 **B와 동일한** int4 tile-packed로 export.
- **왜**: PTQ(B)의 양자화 손실을 **되돌린다**. 서빙 포맷·크기가 B와 **완전히 동일**(5.77GB)하므로 B와의 차이는
  **순수하게 train-aware 효과**.
- **어떻게(W2 교정)**: matched fake-quant(`Int4WeightOnlyConfig(g128)`에서 추론 → tile-packed 서빙과 **동일 int4 family**)
  삽입 → 양자화 대상 linear만 STE로 **600 step**(v1의 1.5×) 재학습(8-bit Adam+grad ckpt로 단일 A100 적합) →
  convert → tile-packed int4 export. `v2_run selftest`가 prepare-fires·same-family·convert-roundtrip를 사전 게이트.

### 0) 부트스트랩 & 설정 (재현성)

In [1]:
import os, sys, json, glob
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path: sys.path.insert(0, cand)
        os.chdir(cand); break
print("cwd:", os.getcwd())

cwd: /home/azureuser/work/pdf_qa_extraction


In [2]:
from quantization.data_korquad import load_config
import quantization.v2_pipeline as V
cfg = load_config()
SEED = 42  # representative seed for the demo; metrics below aggregate all seeds
BASE = cfg["base_model"]["selected"]
print("base:", BASE, "| seeds:", cfg.get("seeds"), "| eval held-out:", cfg["data"]["eval_size"])

base: Qwen/Qwen3-8B | seeds: [42, 43, 44] | eval held-out: 1000


In [3]:
import platform, torch, transformers, torchao
env = {"python": platform.python_version(), "torch": torch.__version__,
       "transformers": transformers.__version__, "torchao": torchao.__version__,
       "cuda": torch.version.cuda,
       "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")}
os.makedirs("quantization/results", exist_ok=True)
json.dump(env, open("quantization/results/env_C.json", "w"), ensure_ascii=False, indent=2)
env

/home/azureuser/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.10.12',
 'torch': '2.11.0+cu130',
 'transformers': '4.57.6',
 'torchao': '0.17.0',
 'cuda': '13.0',
 'device': 'NVIDIA A100 80GB PCIe'}

### 1) 산출물 위치 (per-seed 아티팩트 — 학습/양자화는 `v2_run`으로 이미 실행됨)

In [4]:
MDIR = f"quantization/artifacts/C_int4_qat_seed{SEED}"
print("int4 QAT:", MDIR, "exists:", os.path.isdir(MDIR))

int4 QAT: quantization/artifacts/C_int4_qat_seed42 exists: True


**QAT 학습 로그(실측)** — 3 seed의 loss/크기:

In [5]:
for f in sorted(glob.glob("quantization/results/C_train_seed*.json")):
    s = f.split("seed")[-1].split(".")[0]; l = json.load(open(f))
    print(f"seed {s}: train_loss={l['train_loss']:.4f}  size_gb={l['size_gb']}")

seed 42: train_loss=0.0058  size_gb=5.7705
seed 43: train_loss=0.0157  size_gb=5.7705
seed 44: train_loss=0.0148  size_gb=5.7705


### 2) 동작 데모 (필수) — held-out KorQuAD 질문 **1개** 실측 생성

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
data = V.load_slices(cfg); ex = data["eval"][0]
tok = AutoTokenizer.from_pretrained(MDIR)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "left"; tok.truncation_side = "left"
model = AutoModelForCausalLM.from_pretrained(MDIR, device_map="cuda")  # int4 dir carries its own config
prompt = V.build_chat_prompt(tok, V.system_prompt(cfg), ex.context, ex.question, None,
                             V.enable_thinking_flag(cfg), True)
enc = tok(prompt, return_tensors="pt", truncation=True, max_length=3072).to(model.device)
out = model.generate(**enc, max_new_tokens=cfg["eval"]["max_new_tokens"], do_sample=False,
                     pad_token_id=tok.pad_token_id)
ans = V.extract_answer(tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))
print("[문맥]", ex.context[:120], "...")
print("[질문]", ex.question)
print("[정답]", ex.answers)
print("[C_int4_qat 모델답]", ans)
del model; torch.cuda.empty_cache()

The tokenizer you are loading from 'quantization/artifacts/C_int4_qat_seed42' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:32<00:32, 32.86s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:40<00:00, 17.80s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:40<00:00, 20.06s/it]


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[문맥] 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도  ...
[질문] 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[정답] ['대중교통체계']
[C_int4_qat 모델답] 대중교통체계


### 3) 수치 — 3 seed 평균±표준편차 (EM/F1/ppl · `results/three_way_table.json`)

In [7]:
tw = json.load(open("quantization/results/three_way_table.json"))
ag = tw["aggregate"]["C_int4_qat"]
print("C_int4_qat  base=", ag["base_model"], " seeds=", ag["seeds"], " n_eval=", ag["n_eval"])
print("  F1  = %.3f +/- %.3f" % (ag["f1"]["mean"], ag["f1"]["std"]))
print("  EM  = %.3f +/- %.3f" % (ag["exact_match"]["mean"], ag["exact_match"]["std"]))
print("  ppl = %.3f +/- %.3f" % (ag["perplexity"]["mean"], ag["perplexity"]["std"]))
print("  size_gb =", ag.get("size_gb"), " tok/s(eager) = %.2f" % ag["tok_per_s"]["mean"])

C_int4_qat  base= Qwen/Qwen3-8B  seeds= [42, 43, 44]  n_eval= 1000
  F1  = 94.819 +/- 0.108
  EM  = 87.567 +/- 0.340
  ppl = 10.688 +/- 0.197
  size_gb = 5.7705  tok/s(eager) = 4.46


### 4) 3-way 종합 (A > C > B) — 평균±표준편차 + train-aware 회복
A(품질 상한) · B(PTQ, 양자화 손실) · C(QAT, 손실 회복). B·C는 서빙 포맷 동일(5.77GB).

In [8]:
rows = []
for m in ("A_bf16", "B_int4_ptq", "C_int4_qat"):
    a = tw["aggregate"][m]
    rows.append((m, a["exact_match"]["mean"], a["exact_match"]["std"],
                 a["f1"]["mean"], a["f1"]["std"], a["perplexity"]["mean"], a.get("size_gb")))
print("%-14s %-14s %-16s %-14s %s" % ("method", "EM", "F1", "ppl", "size_gb"))
for m, em, es, f1, fs, pp, sz in rows:
    print("%-14s %6.2f±%-5.2f %7.3f±%-6.3f %7.3f     %s" % (m, em, es, f1, fs, pp, sz))
recov = tw["aggregate"]["C_int4_qat"]["f1"]["mean"] - tw["aggregate"]["B_int4_ptq"]["f1"]["mean"]
print("\nQAT가 PTQ 대비 회복한 F1: +%.3f (서빙 포맷·크기 동일 → 순수 train-aware 효과)" % recov)

method         EM             F1               ppl            size_gb
A_bf16          87.80±0.51   94.830±0.187    9.050     15.2713
B_int4_ptq      86.40±0.28   94.190±0.232   10.094     5.7705
C_int4_qat      87.57±0.34   94.819±0.108   10.688     5.7705

QAT가 PTQ 대비 회복한 F1: +0.629 (서빙 포맷·크기 동일 → 순수 train-aware 효과)


### 5) 서빙 & 처리량
B·C의 int4 tile-packed 아티팩트는 **vLLM**으로 서빙된다. 동일조건 tok/s(배치 스윕 + TTFT/p50/p99)는 `v2_bench.py` → `results/vllm_throughput.json` 참고(단일 tok/s 소스, W3·W4 교정).